In [2]:
import os
import random
import sys
import time
import warnings
from pathlib import Path

import mlflow
import numpy as np
import optuna
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.wtd_har_wrapper import HARWTDWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

mlflow.set_tracking_uri("sqlite:///mlflow.db")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "wtd_har"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- La optimización bajo validación Walk-Forward seleccionó al WTD-HAR con ventanas de 5 días (semanal), 22 días (mensual) y nivel de descomposición wavelet $J = 3$ como la mejor configuración.
- El filtrado causal mediante transformada wavelet discreta atenuó el ruido de microestructura de los retornos al cuadrado, mejorando la pérdida QLIKE fuera de muestra frente al modelo HAR ($0.4084$ vs. $0.5076$) y alcanzando un desempeño equiparable al del GARCH ($0.4097$).
- En el conjunto de prueba, el preprocesamiento wavelet absorbió la autocorrelación serial y los efectos ARCH residuales, mientras los errores se ajustaron a una distribución t-Student ($\nu \approx 6.634$) bajo el esquema recursivo un paso adelante.
- A diferencia del modelo HAR, la prueba BDS detectó estructura no lineal remanente en los residuos fuera de muestra, lo que justifica la transición hacia arquitecturas híbridas no lineales como WTD-HAR-GRU.
- Al igual que el resto de las especificaciones paramétricas evaluadas, el modelo conservó un efecto de suavizado que subestimó la magnitud del pico extremo de volatilidad de 2026.
:::

## Introducción

El modelo WTD-HAR mantiene paridad experimental con el resto del *benchmark* (Pan et al., 2025): recibe la serie de retornos logarítmicos diarios ($r_t$) y pronostica la varianza de Rogers-Satchell $\left(\widehat{RS}_t\right)$ mediante una estructura de rezagos multiescala.

A diferencia de estimar el HAR sobre $r_{t-i}^2$, esta variante aplica una transformada *wavelet* discreta causal (`sym4`, extensión simétrica `symmetric`) sobre los retornos al cuadrado ($r_t^2$) para atenuar el ruido de alta frecuencia y los picos transitorios antes de formar las escalas temporales, mejorando así la estabilidad de las señales rezagadas y el ajuste del modelo. En cada instante $t$, se toma la ventana causal hasta $t$ de longitud efectiva $W = \max(L_{\min}, 2L_{\min})$ con $L_{\min} = (L_{\text{dec}} - 1)2^J$, aplicando relleno por borde cuando la historia disponible es menor que $L_{\min}$.

Sobre esa ventana se calcula la descomposición `wavedec` hasta el nivel $J$ (`wtd_level`), y el nivel de ruido sobre el coeficiente de detalle más fino se estima mediante:

$$
\hat{\sigma} = \frac{\text{median}(|d_1|)}{0.6745}
$$

Con esta estimación de ruido, se aplica umbralización suave (`soft`) con umbral universal $\lambda = \hat{\sigma}\sqrt{2\ln n}$ a los coeficientes de detalle y se reconstruye la serie con `waverec` conservando únicamente el último valor ($\tilde{v}_t = \hat{x}_t$), evitando que el suavizado distorsione el ajuste de largo plazo al quedarse solo con el punto causal más reciente.

A partir de la serie filtrada $\tilde{v}_t$, se construyen los regresores diarios, semanales $\left(L_w\right)$ y mensuales $\left(L_m\right)$ con rezago `shift(1)` para evitar fuga de información:

$$
\widehat{RS}_t = c + \beta_d \tilde{v}_{t-1} + \beta_w \left(\frac{1}{L_w}\sum_{i=1}^{L_w} \tilde{v}_{t-i}\right) + \beta_m \left(\frac{1}{L_m}\sum_{i=1}^{L_m} \tilde{v}_{t-i}\right) + \varepsilon_t
$$

La estimación puntual de $\boldsymbol{\beta}$ se obtiene mediante factorización QR reducida, con respaldo en mínimos cuadrados (`lstsq`) ante mal condicionamiento, e inferencia robusta HAC/Newey-West (además de intervalos por *block bootstrap* cuando se solicitan). Los supuestos y limitaciones del modelo HAR se mantienen según lo declarado en su libreta de referencia, aplicados ahora sobre la serie suavizada.

## Entrenamiento

Para la selección de hiperparámetros del modelo WTD-HAR, se aplica el mismo protocolo experimental detallado en las secciones anteriores. Se ejecuta una réplica bajo el esquema *walk-forward*, utilizando optimización bayesiana (Optuna) para explorar las longitudes de las ventanas temporales semanal ($L_w$) y mensual ($L_m$), junto con el nivel de descomposición *wavelet* ($J$).

Se mantienen los rangos de rezagos del modelo HAR ($L_w \in \{5, 7\}$ y $L_m \in \{22, 30\}$) y se incorpora el hiperparámetro `wtd_level` sobre el conjunto $\{1, 2, 3\}$, utilizando la *wavelet* madre `sym4` y fijando un máximo de $J = 3$ niveles de descomposición conforme a la práctica reportada en la literatura de referencia (Pan et al., 2025). Con ello, el espacio discreto abarca 12 combinaciones posibles ($2 \times 2 \times 3$).

Se conservan los criterios de control computacional (límite de 50 evaluaciones y parada temprana tras 15 intentos). Dado que la estimación de los coeficientes se resuelve de forma determinista mediante factorización QR reducida (con respaldo `lstsq`), no existen fallos de convergencia iterativa como ocurre en optimizaciones por máxima verosimilitud, y la parada temprana suele activarse una vez exploradas las combinaciones relevantes del espacio de búsqueda. Tampoco se aplica factor de escalado adicional sobre los retornos ni sobre la varianza objetivo, preservando la coherencia con el resto del *benchmark* y la interpretabilidad de los parámetros.

In [3]:
def wtd_har_search_space(trial: optuna.Trial) -> dict:
    return {
        "lag_weekly": trial.suggest_categorical("lag_weekly", [5, 7]),
        "lag_monthly": trial.suggest_categorical("lag_monthly", [22, 30]),
        "wtd_level": trial.suggest_categorical("wtd_level", [1, 2, 3]),
        "seed": 42,
    }


exec_id = 6
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=HARWTDWrapper,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=wtd_har_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

  0%|          | 0/50 [00:00<?, ?it/s]

La búsqueda identificó como configuración óptima al WTD-HAR con $L_w=5$, $L_m=22$ y nivel *wavelet* $J=3$ $\left(\text{QLIKE} = 0.5218\right)$, completando 25 ensayos sin descartes por poda (0 podados). Esto indica que la estimación determinista mediante factorización QR reducida y el filtrado *wavelet* causal mantienen estabilidad numérica desde la primera partición.

El entrenamiento bajo el esquema *walk-forward* acumuló 42.05 segundos en el ajuste por partición, con un tiempo medio de 0.7010 segundos por partición (y un avance de 25 de 50 ensayos en 3 minutos y 7 segundos a ~3.88 s por iteración en el bucle de Optuna). Este costo mantiene un rango moderado y permite su estimación y actualización periódica sin requerir hardware especializado.

In [5]:
resumen_data = []

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
best_params_str = "-"
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="HARWTDWrapper_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

        w_val = study.best_params.get("lag_weekly", "?")
        m_val = study.best_params.get("lag_monthly", "?")
        j_val = study.best_params.get("wtd_level", "?")
        best_params_str = f"({w_val}, {m_val}, {j_val})"

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor Config (Lw, Lm, J)": best_params_str,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,"Mejor Config (Lw, Lm, J)",Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
25,25,0,"(5, 22, 3)",0.5218,42.0500,0.7010


El análisis de las configuraciones evaluadas del WTD-HAR muestra que el desempeño se concentra en la ventana mensual de 22 días y en el nivel de descomposición *wavelet* $J=3$, encabezando los registros $(5, 22, 3)$ y $(7, 22, 3)$ con un QLIKE medio de $0.522 \pm 0.112$ frente a los niveles intermedios con $J=1$ ($0.542\text{--}0.545$) y $J=2$ ($0.565 \pm 0.132$). Esta reducción frente al HAR base (cuya mejor configuración alcanzó $0.729 \pm 0.212$) se puede deber a que la transformada *wavelet* causal (`sym4`) con umbralización suave atenúa el ruido de alta frecuencia y las perturbaciones transitorias en los rezagos de volatilidad, preservando con mayor estabilidad la señal persistente asociada a la escala mensual de 22 días sin requerir el ajuste de la ventana semanal a 7 días que exigía la serie sin filtrar.

In [6]:
exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

df_wtd_har = pd.read_csv(exec_dir / "HARWTDWrapper_results.csv")
cols_agrupar = ["lag_weekly", "lag_monthly", "wtd_level"]
df_agg = df_wtd_har.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()

df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "lag_weekly": "Lag Semanal (Lw)",
    "lag_monthly": "Lag Mensual (Lm)",
    "wtd_level": "Nivel Wavelet (J)",
})

df_final = df_top5[["Lag Semanal (Lw)", "Lag Mensual (Lm)", "Nivel Wavelet (J)", "QLIKE"]]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lag Semanal (Lw),Lag Mensual (Lm),Nivel Wavelet (J),QLIKE
5,22,3,0.522 ± 0.112
7,22,3,0.522 ± 0.112
5,22,1,0.542 ± 0.116
7,22,1,0.545 ± 0.117
7,22,2,0.565 ± 0.132


La elección de $(5, 22, 3)$ sobre $(7, 22, 3)$, dado que ambas configuraciones registran el mismo QLIKE medio y dispersión entre particiones, responde a criterios de correspondencia temporal y alineación con la literatura. Esta decisión se puede deber a que la ventana de 5 días representa la semana completa de negociación bursátil y mantiene coherencia directa con la especificación multiescala de Corsi (2009) y el marco de referencia de Pan et al. (2025) sin necesidad de extender el rezago semanal a 7 días.

In [7]:
best_model = HARWTDWrapper(lag_weekly=5, lag_monthly=22, wtd_level=3)
best_model.fit(y_train, rs_train)

En el WTD-HAR (5, 22, 3), la componente mensual fue estadísticamente significativa ($p = .019$), mientras que las componentes diaria ($p = .079$) y semanal ($p = .067$) superaron ligeramente el umbral habitual en su p-valor, pero sus intervalos de confianza del 95 % por *block bootstrap* no contuvieron el cero, lo que respaldó su efecto positivo sobre la varianza filtrada.

A diferencia del modelo HAR, donde los tres rezagos fueron significativos y la escala mensual dominó en magnitud ($0.4027$ frente a $0.3478$ diario y $0.0989$ semanal), en el WTD-HAR los coeficientes diario ($0.3651$) y mensual ($0.3653$) presentaron una magnitud similar y el semanal se situó en $0.1258$. Esta diferencia se puede deber a que la descomposición *wavelet* causal (`sym4`, $J=3$) depuró el ruido de alta frecuencia de la serie de entrada y redistribuyó la persistencia entre el rezago diario y el mensual.

In [8]:
df_params_boot = best_model.get_parameter_table(
    B=2000, decimals=4, block_size=22
)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior (95%)",
    "CI Upper": "IC Superior (95%)"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior (95%),IC Superior (95%)
const,0.0000,0.0001,.716,-0.0001,0.0001
RV_d,0.3651,0.2077,.079,0.3476,0.3851
RV_w,0.1258,0.0686,.067,0.0643,0.1945
RV_m,0.3653,0.1551,.019,0.2800,0.4718


La suma de los tres coeficientes ($0.8560$) resultó inferior a la unidad. En el marco de los modelos autorregresivos, este resultado implicó que los choques de volatilidad presentaron persistencia, pero tendieron a disiparse en el largo plazo (reversión a la media). Asimismo, los intervalos de confianza al 95 %, calculados mediante *bootstrapping* de bloques, no incluyeron el cero en ninguna de las tres componentes, respaldando la estabilidad de los estimadores.

A partir de las varianzas predichas y los retornos reales del conjunto de entrenamiento, se construyeron los residuos estandarizados en muestra mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

Al igual que en el modelo HAR base, los contrastes de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) rechazaron la hipótesis nula (*p* < .05), junto con las pruebas de normalidad (Jarque-Bera y Lilliefors). Esto indicó que la estructura lineal del modelo sobre la serie filtrada tampoco absorbió por completo la dependencia serial, la heterocedasticidad condicional ni la no linealidad de los retornos.

Esta persistencia residual se puede deber a dos mecanismos: (i) el suavizado introducido por el filtrado *wavelet* causal y las ventanas móviles (1/5 y 1/22), o bien (ii) la agregación mediante promedios móviles con pesos fijos y la ausencia de actualización recursiva de la varianza condicional. Dado que este mismo comportamiento se observó en el modelo HAR, resulta más probable que la estructura lineal basada en promedios móviles no captura por completo las dinámicas condicionales y no lineales presentes en los retornos.

In [9]:
residuos_wtd_har = best_model.model_fit.resid
std_residuals = residuos_wtd_har / residuos_wtd_har.std()

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),138.7365,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),236.3644,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),403.3363,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),221.1365,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),239.7304,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),375.0609,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,20828910.0559,< .001,--,Residuos No Normales
Lilliefors,0.3084,< .001,--,Residuos No Normales
BDS (Dim 2),16.3167,< .001,--,Estructura No Lineal Significativa


El análisis de la distribución de los residuos evidenció que la estructura del WTD-HAR tampoco capturó la normalidad de las innovaciones. La inspección visual del histograma reveló una distribución altamente leptocúrtica, caracterizada por una concentración pronunciada alrededor de cero y colas pesadas, lo que se confirmó al rechazar la hipótesis de normalidad mediante las pruebas de Jarque-Bera y Lilliefors ($p < .001$).

A diferencia del modelo HAR base, donde también se rechazó el ajuste a una distribución *t*-Student, en el WTD-HAR los contrastes de Kolmogorov-Smirnov ($p = .496$) y Cramér-von Mises ($p = .374$) no rechazaron una distribución *t*-Student ajustada con grados de libertad cercanos a uno ($\nu \approx 1.20$), próxima a una distribución de Cauchy. Un grado de libertad cercano a uno colas extremadamente pesadas y varianza teórica indefinida o infinita ($\nu \le 2$), lo que indica que el ajuste se logra concentrando masa en el pico central y absorbiendo valores extremos en las colas, pero sin equivaler a una distribución *t* bien comportada de varianza finita; en el gráfico Q-Q persisten además desviaciones en los cuantiles extremos. Esta forma se puede deber a que el filtrado *wavelet* causal (`sym4`, $J=3$) atenúa el ruido de alta frecuencia de la serie de entrada y suaviza las fluctuaciones moderadas, dejando un cuerpo central agudo acompañado de realizaciones extremas.

In [16]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 3),
    legend_loc="upper left"
)

La inspección visual del ajuste en la muestra de entrenamiento mostró que el WTD-HAR (5, 22, 3) siguió la trayectoria general y los ciclos de la volatilidad realizada sin desfases temporales evidentes, manteniendo una dinámica similar a la del modelo HAR. Como se observa en el recuadro de la crisis por COVID-19, el modelo subestimó los picos más extremos de volatilidad y suavizó las oscilaciones intradiarias o de muy corto plazo. Esta similitud frente al HAR base y el subajuste en los choques de mayor intensidad se pueden deber a que la agregación lineal en ventanas rezagadas conserva una respuesta moderada ante eventos abruptos, reflejándose en las colas pesadas y la estructura leptocúrtica observadas en los residuos.

In [10]:
pred_varianza_wtd_har = best_model.model_fit.fittedvalues

lag = best_model.lag_monthly
rs_volatilidad_real = np.sqrt(rs_train[lag:])
pred_volatilidad_wtd_har = np.sqrt(pred_varianza_wtd_har)
fechas_alineadas = df_train.index[lag:]

mask_crisis = (fechas_alineadas >= "2020-01-01") & (fechas_alineadas <= "2020-06-30")

ts.plot_volatility_fit(
    dates=fechas_alineadas,
    rs_variance=rs_volatilidad_real,
    pred_variance=pred_volatilidad_wtd_har,
    title="Ajuste dentro la Muestra de Entrenamiento (WTD-HAR 5,22,3)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad WTD-HAR $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del WTD-HAR sigue el mismo protocolo experimental detallado en las secciones anteriores. El rendimiento se contrasta comparando el ajuste en entrenamiento con la capacidad predictiva fuera de muestra sobre datos no vistos.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se mantiene el criterio de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Al igual que en los modelos anteriores, un valor de $|\Delta_{abs}|$ que supere el umbral de tolerancia de 0.05 evidencia falta de generalización.

Las predicciones fuera de muestra se generan ejecutando el método `forecast` bajo un esquema de ventana expansiva (*walk-forward* expansivo), aplicando en cada paso el filtrado *wavelet* causal (`sym4`, `wtd_level` $= J$) y reestimando los coeficientes mediante factorización QR reducida para evitar cualquier fuga de información. Tampoco se aplican factores de escalado adicionales a los retornos ni a la varianza objetivo, por lo que las varianzas pronosticadas se evalúan directamente en su escala original frente a la varianza de Rogers-Satchell.

In [17]:
start_time = time.time()

preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)
end_time = time.time()
tiempo_evaluacion = end_time - start_time

df_preds = pd.DataFrame({
    "date": df_test.index,
    "rs_true": rs_test,
    "y_pred": preds_test,
})

output_path = RESULTS_PATH / "best_wtd_har_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva, con filtrado *wavelet* causal y reestimación por factorización QR reducida en cada paso, tuvo una duración total de 13.43 segundos (~0.014 segundos por predicción sobre las 953 observaciones del conjunto de prueba). Este tiempo es inferior al del HAR base (20.26 segundos) y confirma la eficiencia computacional del enfoque paramétrico frente a arquitecturas de *Deep Learning* reentrenadas iterativamente.

El análisis de los residuos fuera de muestra mostró un comportamiento inverso al del HAR base sobre el *holdout*. En el WTD-HAR, las pruebas de Ljung-Box y ARCH-LM no rechazaron la hipótesis nula en ningún rezago, indicando ausencia de autocorrelación serial y de efectos ARCH residuales, mientras que la prueba BDS detectó estructura no lineal significativa. A diferencia del modelo HAR, donde persistieron los efectos ARCH pero no se rechazó la ausencia de no linealidad, el filtrado *wavelet* en WTD-HAR parece absorber la heterocedasticidad condicional pero deja no linealidad remanente en los residuos, lo que justifica explorar extensiones híbridas no lineales como WTD-HAR-GRU.

In [18]:
df_preds = pd.read_csv(RESULTS_PATH / "best_wtd_har_predictions.csv")

std_residuals_oos = y_test / np.sqrt(df_preds["y_pred"].to_numpy(dtype=float))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),9.3748,.497,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),23.1613,.281,.843,Sin Autocorrelación
Ljung-Box (Lag 30),27.0786,.619,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),9.7457,.463,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),14.5373,.802,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),18.3002,.953,1.000,Sin Efectos ARCH
Jarque-Bera,169.1606,< .001,--,Residuos No Normales
Lilliefors,0.0400,.002,--,Residuos No Normales
BDS (Dim 2),-2.1201,.034,--,Estructura No Lineal Significativa


La distribución empírica de los residuos estandarizados del WTD-HAR sigue de cerca la curva *t*-Student ($\nu = 6.6341$), y el gráfico Q-Q muestra una alineación estrecha entre los cuantiles muestrales y los teóricos en el centro y buena parte de las colas, con pequeñas desviaciones en los extremos. A diferencia del entrenamiento, donde los grados de libertad estimados eran cercanos a uno ($\nu \approx 1.20$) y reflejaban colas mucho más pesadas ante episodios de volatilidad severa, en prueba el valor de $\nu = 6.6341$ indica colas más moderadas y regulares.

In [24]:
_, _ = tst.plot_distribution_fit(
    pd.Series(z_oos_clean),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.6),
    legend_loc="upper left"
)

Los residuos estandarizados del WTD-HAR rechazaron la normalidad, mientras que las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .647$; Cramér-von Mises: $p = .585$) no rechazaron una distribución *t*-Student. Este resultado concuerda con la prueba de razón de verosimilitud (LRT: estadístico $= 46.61$, $p < .001$) y con una reducción de $\Delta \text{AIC} = 44.60$ y $\Delta \text{BIC} = 39.75$ a favor de la *t*-Student frente a la normal. Este patrón coincide con el del modelo HAR sobre el *holdout*, donde la distribución *t*-Student con grados de libertad moderados (en torno a $\nu \approx 6$) también superó a la normal, lo que indica que el modelo representa de forma adecuada las innovaciones fuera de muestra pese a las limitaciones frente a choques extremos en entrenamiento.

Este cambio entre entrenamiento y prueba puede deberse principalmente al esquema de predicción un paso adelante (*one-step-ahead*) con actualización recursiva, ya que reestimar y pronosticar de forma secuencial sobre ventanas más recientes ajusta mejor la escala condicional local que un ajuste estático de largo plazo sobre crisis severas y heterogéneas.

In [25]:
_= tst.distribution_fit_table(
    pd.Series(z_oos_clean),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,456.46","2,916.91","2,926.63",-,-
t-Student,"-1,433.15","2,872.31","2,886.88",46.61,< .001


El modelo **WTD-HAR** cumplió con el criterio de tolerancia en la métrica principal QLIKE ($\Delta \le 0.05$) y mostró una mejora clara tanto en entrenamiento como en prueba frente al modelo HAR  ($0.5076$ en prueba), alcanzando un desempeño equiparable al del modelo GARCH ($0.4084$ frente a $0.4097$ en prueba). Esto indica que el filtrado *wavelet* de los componentes autorregresivos atenúa el ruido de microestructura y mejora la generalización fuera de muestra frente a la especificación lineal sin filtrar, situando a **WTD-HAR** al nivel de la referencia GARCH. El incremento acotado del QLIKE entre entrenamiento y prueba, en contraste con las variaciones marginales de las métricas puntuales y probabilísticas, responde a la asimetría de esta función de pérdida, que penaliza con mayor severidad la subestimación de la varianza en episodios de alta volatilidad.

Las métricas ajustadas por heterocedasticidad aumentaron en prueba ($+1.6191$ en HMSE y $+0.0402$ en HMAE), aunque en menor magnitud que en el HAR base ($+3.0418$ y $+0.0753$), un comportamiento coherente con la ponderación inversa por la varianza, que amplifica los desvíos relativos durante los días de mayor turbulencia.

In [19]:
df_preds = pd.read_csv(RESULTS_PATH / "best_wtd_har_predictions.csv")

preds_train_var = best_model.model_fit.fittedvalues
preds_test_var = df_preds["y_pred"].to_numpy(dtype=float)
rs_test_aligned = df_preds["rs_true"].to_numpy(dtype=float)

lag = best_model.lag_monthly
rs_train_aligned = rs_train[lag:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test_aligned,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.3738,0.4084,+0.0345
CRPS,0.0127,0.0124,-0.0004
Pinball (τ=0.05),0.0028,0.0026,-0.0001
Pinball (τ=0.95),0.0013,0.0015,+0.0002
MSE,0.0000,0.0000,+0.0000
RMSE,0.0017,0.0022,+0.0005
MAE,0.0004,0.0004,+0.0000
HMSE,1.5707,3.1898,+1.6191
HMAE,0.6915,0.7317,+0.0402


La inspección visual de las predicciones evidenció que el WTD-HAR acompaña la trayectoria general de la volatilidad realizada, pero reacciona con rezago y no logra alcanzar la magnitud del pico extremo de 2026.

La incapacidad para ajustar el shock de volatilidad de 2026 también se observó en los demás modelos paramétricos evaluados hasta el momento, ya que tanto el HAR como el GARCH muestran respuestas atenuadas o tardías ante el máximo registrado y subestiman los choques abruptos. Aunque las especificaciones basadas en volatilidad realizada exhiben mayor reactividad local tras el salto frente a la inercia del GARCH, ninguna logra capturar la amplitud total del evento, lo que explica la penalización en QLIKE y respalda la incorporación de estructuras no lineales capaces de modelar cambios abruptos de régimen.

In [27]:
df_preds["date"] = pd.to_datetime(df_preds["date"])

rs_test_vol = np.sqrt(df_preds["rs_true"])
pred_test_vol = np.sqrt(df_preds["y_pred"])

mask_crisis = (df_preds["date"] >= "2026-02-01") & (df_preds["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (WTD-HAR 5, 22, 3)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad WTD-HAR $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis if mask_crisis.any() else None,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La evaluación del modelo WTD-HAR registró una pérdida QLIKE en el conjunto de prueba de 0.4084, lo que representó una mejora frente al modelo HAR (0.5076) y situó su desempeño al nivel del mejor modelo paramétrico evaluado hasta el momento, el GARCH (0.4097). La optimización seleccionó ventanas de 5 días semanales, 22 días mensuales y un nivel de descomposición wavelet $J = 3$, confirmando que el filtrado causal de los retornos al cuadrado atenúa el ruido de microestructura y mejora el desempeño predictivo de la estructura heterogénea de Corsi (2009) sobre la volatilidad del Brent.

El modelo presentó un desempeño adecuado durante los periodos de estabilidad y redujo las métricas ajustadas por heterocedasticidad frente al HAR base, pero conservó un efecto de suavizado ante eventos extremos. Al igual que en el resto de las especificaciones paramétricas analizadas, la trayectoria pronosticada reaccionó con rezago y no alcanzó la amplitud del pico de volatilidad registrado en 2026. Debido a la penalización asimétrica de la función QLIKE, esta subestimación puntual concentró gran parte del error fuera de muestra.

El diagnóstico de residuos confirmó que el preprocesamiento wavelet absorbió la autocorrelación serial y los efectos ARCH residuales fuera de muestra, mostrando además un ajuste adecuado a una distribución t-Student ($\nu \approx 6.634$) bajo el esquema recursivo un paso adelante. No obstante, a diferencia del HAR estándar, la prueba BDS detectó estructura no lineal remanente en los errores de prueba. Esta dependencia no lineal no capturada, sumada a la eficiencia computacional del enfoque y a su superioridad frente al HAR base, justifica la transición hacia arquitecturas híbridas no lineales como WTD-HAR-GRU para modelar discontinuidades abruptas y mejorar la capacidad predictiva.